In [1]:
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

In [3]:
ip_file = "metadata/arxiv-metadata-oai-snapshot.json"
op_file = "metadata/arxiv_filtered_master.parquet"
target_cats = { 'cs.AI','cs.LG','cs.NE','cs.CL','cs.CV','stat.ML','cs.IR','cs.GT'}

In [7]:
i=0
with open(ip_file,'r') as f:
    for line in f:
        doc= json.loads(line)
        i+=1
        if i==10:
            break

        print(doc)

{'id': '0704.0001', 'submitter': 'Pavel Nadolsky', 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'comments': '37 pages, 15 figures; published version', 'journal-ref': 'Phys.Rev.D76:013009,2007', 'doi': '10.1103/PhysRevD.76.013009', 'report-no': 'ANL-HEP-PR-07-12', 'categories': 'hep-ph', 'license': None, 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from the Fermilab

In [17]:
def filter_arxiv_dump():
    print(f"reading {ip_file}")

    batch_data=[]
    batch_size = 50000
    total_papers=0
    kept_papers=0

    schema = pa.schema([
    ('id',pa.string()),
    ('title', pa.string()),
    ('abstract',pa.string()),
    ('year',pa.int32()),
    ('categories',pa.string()),
    ('version',pa.string())
    ])

    writer = None

    with open(ip_file,'r') as f:
        for line in tqdm(f, desc='Processing Lines'):
            try:
                doc = json.loads(line)
                total_papers +=1

                doc_cats = set(doc['categories'].split(' '))
                # print(doc_cats)
                # if total_papers == 10:
                #     break
                if not doc_cats.isdisjoint(target_cats):

                    try:
                        v1_date = doc['versions'][0]['created']
                        year = int(v1_date.split(' ')[3])
                    except:
                        year = 0

                    batch_data.append({
                    'id':doc['id'],
                    'title': doc['title'].replace('\n', ' ').strip(),
                    'abstract': doc['abstract'].replace('\n', ' ').strip(),
                    'categories': doc['categories'],
                    'year': year,
                    'versions': json.dumps(doc['versions'])
                    })
                    kept_papers+=1

                if len(batch_data)>=batch_size:
                    table = pa.Table.from_pylist(batch_data,schema=schema)
                    if writer is None:
                        writer = pq.ParquetWriter(op_file,schema)
                    writer.write_table(table)
                    batch_data=[] #clear ram

            except Exception as e:
                print(e)
                continue 

    if batch_data:
        table = pa.Table.from_pylist(batch_data, schema=schema)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_FILE, schema)
        writer.write_table(table)
        
    if writer:
        writer.close()

    print(f"\n✅ Done! Scanned {total_papers} papers.")
    print(f"🎯 Kept {kept_papers} relevant papers.")
    print(f"💾 Saved to {op_file}")
        
        

                

In [18]:
filter_arxiv_dump()

reading metadata/arxiv-metadata-oai-snapshot.json


Processing Lines: 2938427it [00:24, 122036.10it/s]



✅ Done! Scanned 2938427 papers.
🎯 Kept 542924 relevant papers.
💾 Saved to arxiv_filtered_master.parquet


In [19]:
import pandas as pd

df_check = pd.read_parquet("arxiv_filtered_master.parquet")
print("Year Distribution:")
print(df_check['year'].value_counts().sort_index())

# Check for the missing QLoRA paper by its known ID
print("\nChecking for QLoRA (2305.14314):")
print(df_check[df_check['id'] == '2305.14314'])

Year Distribution:
year
1993         6
1994       237
1995       252
1996       230
1997       184
1998       160
1999       111
2000       283
2001       168
2002       263
2003       262
2004       306
2005       326
2006       403
2007       582
2008       859
2009      1250
2010      1740
2011      2403
2012      3934
2013      5142
2014      5538
2015      7265
2016     10907
2017     15968
2018     24080
2019     34248
2020     45677
2021     50270
2022     55366
2023     69302
2024     89604
2025    109893
2026      5705
Name: count, dtype: int64

Checking for QLoRA (2305.14314):
                id                                          title  \
288962  2305.14314  QLoRA: Efficient Finetuning of Quantized LLMs   

                                                 abstract  year categories  \
288962  We present QLoRA, an efficient finetuning appr...  2023      cs.LG   

       version  
288962    None  


In [23]:
df_check['id']

0                0704.0047
1                0704.0050
2                0704.0304
3                0704.0671
4                0704.0954
                ...       
542919    quant-ph/0609117
542920    quant-ph/0611234
542921    quant-ph/0702072
542922    quant-ph/9802028
542923    quant-ph/9907009
Name: id, Length: 542924, dtype: object